# Cloud Native Satellite Data Demo 1
* Use pystac_client to search the STAC API at Planetary Computer
* Use odc.stac to load the data
* Use hvplot to visualize the data

In [1]:
import pystac_client
import planetary_computer
from rich.table import Table
import hvplot.xarray

# Connect to the Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [2]:
collections = list(catalog.get_all_collections())
collections.sort(key=lambda c: c.id)
table = Table("ID", "Title", title="Planetary Computer collections")
for collection in collections:
    table.add_row(collection.id, collection.title)
table

                                          Planetary Computer collections                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                                                     ┃ Title                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 3dep-lidar-classification                              │ USGS 3DEP Lidar Classification                         │
│ 3dep-lidar-copc                                        │ USGS 3DEP Lidar Point Cloud                            │
│ 3dep-lidar-dsm                                         │ USGS 3DEP Lidar Digital Surface Model                  │
│ 3dep-lidar-dtm                                         │ USGS 3DEP Lidar Digital Terrain Model                  │
│ 3dep-lidar-dtm-native                                  │ USGS 3DEP Lidar Digital Terrain Model (Native)         │
│ 3dep-lidar-hag                                         │ USGS 3DEP Lidar Height above Ground                    │
│ 3dep-lidar-intensity                                   │ USGS 3DEP Lidar Intensity                              │
│ 3dep-lidar-pointsourceid                               │ USGS 3DEP Lidar Point Source                           │
│ 3dep-lidar-returns                                     │ USGS 3DEP Lidar Returns                                │
│ 3dep-seamless                                          │ USGS 3DEP Seamless DEMs                                │
│ alos-dem                                               │ ALOS World 3D-30m                                      │
│ alos-fnf-mosaic                                        │ ALOS Forest/Non-Forest Annual Mosaic                   │
│ alos-palsar-mosaic                                     │ ALOS PALSAR Annual Mosaic                              │
│ aster-l1t                                              │ ASTER L1T                                              │
│ chesapeake-lc-13                                       │ Chesapeake Land Cover (13-class)                       │
│ chesapeake-lc-7                                        │ Chesapeake Land Cover (7-class)                        │
│ chesapeake-lu                                          │ Chesapeake Land Use                                    │
│ chloris-biomass                                        │ Chloris Biomass                                        │
│ cil-gdpcir-cc-by                                       │ CIL Global Downscaled Projections for Climate Impacts  │
│                                                        │ Research (CC-BY-4.0)                                   │
│ cil-gdpcir-cc-by-sa                                    │ CIL Global Downscaled Projections for Climate Impacts  │
│                                                        │ Research (CC-BY-SA-4.0)                                │
│ cil-gdpcir-cc0                                         │ CIL Global Downscaled Projections for Climate Impacts  │
│                                                        │ Research (CC0-1.0)                                     │
│ conus404                                               │ CONUS404                                               │
│ cop-dem-glo-30                                         │ Copernicus DEM GLO-30                                  │
│ cop-dem-glo-90                                         │ Copernicus DEM GLO-90                                  │
│ daymet-annual-hi                                       │ Daymet Annual Hawaii                                   │
│ daymet-annual-na                                       │ Daymet Annual North America                            │
│ daymet-annual-pr                                       │ Daymet Annual Puerto Rico                              │
│ daymet-daily-hi                                       

In [3]:
# Define our area of interest (AOI) - a rough box around Cape Cod
bbox = [-71.0, 41.5, -69.8, 42.2]

In [4]:
# Let's search for Sentinel-2 Level-2A data from this past summer
# with minimal cloud cover.
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-08-01/2025-09-30",
    query={"eo:cloud_cover": {"lt": 10}}, # less than 10% clouds
)

# Get the results and see what we found
items = search.item_collection()
print(f"Found {len(items)} scenes matching your criteria.")

# Let's inspect the assets of the first item to see the available data bands
if items:
    first_item = items[0]
    print("\nAssets for the first scene:")
    for asset_key, asset in first_item.assets.items():
        print(f"- {asset_key}: {asset.title}")

Found 26 scenes matching your criteria.

Assets for the first scene:
- AOT: Aerosol optical thickness (AOT)
- B01: Band 1 - Coastal aerosol - 60m
- B02: Band 2 - Blue - 10m
- B03: Band 3 - Green - 10m
- B04: Band 4 - Red - 10m
- B05: Band 5 - Vegetation red edge 1 - 20m
- B06: Band 6 - Vegetation red edge 2 - 20m
- B07: Band 7 - Vegetation red edge 3 - 20m
- B08: Band 8 - NIR - 10m
- B09: Band 9 - Water vapor - 60m
- B11: Band 11 - SWIR (1.6) - 20m
- B12: Band 12 - SWIR (2.2) - 20m
- B8A: Band 8A - Vegetation red edge 4 - 20m
- SCL: Scene classfication map (SCL)
- WVP: Water vapour (WVP)
- visual: True color image
- safe-manifest: SAFE manifest
- granule-metadata: Granule metadata
- inspire-metadata: INSPIRE metadata
- product-metadata: Product metadata
- datastrip-metadata: Datastrip metadata
- tilejson: TileJSON with default rendering
- rendered_preview: Rendered preview


In [5]:
# Assuming 'items' is the ItemCollection from the first step
from odc.stac import stac_load


# Load the data using odc-stac.
# Note that we specify the dask chunks as a dictionary.
data_ds = stac_load(
    items,
    bands=["B04", "B03", "B02", "B08"], # B04=Red, B03=Green, B02=Blue, B08=NIR
    bbox=bbox,
    resolution=10,
    chunks={"x": 2048, "y": 2048},
)

# Let's inspect our new xarray Dataset.
# Notice the "Data variables" section.
print(data_ds)

<xarray.Dataset> Size: 13GB
Dimensions:      (y: 7935, x: 10089, time: 10)
Coordinates:
  * y            (y) float64 63kB 4.674e+06 4.674e+06 ... 4.595e+06 4.595e+06
  * x            (x) float64 81kB 3.331e+05 3.331e+05 ... 4.339e+05 4.34e+05
  * time         (time) datetime64[ns] 80B 2025-08-04T15:38:09.024000 ... 202...
    spatial_ref  int32 4B 32619
Data variables:
    B04          (time, y, x) float32 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B03          (time, y, x) float32 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B02          (time, y, x) float32 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B08          (time, y, x) float32 3GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>


In [6]:
import hvplot.xarray  # Make sure to import this to activate the .hvplot accessor
import numpy as np

# --- Step 1: Find the clearest scene (same as before) ---
print("Finding the clearest available scene...")
valid_pixels = data_ds.B02.where(data_ds.B02 > 0).count(dim=["x", "y"])
best_time_slice = valid_pixels.argmax().compute()
best_scene = data_ds.isel(time=best_time_slice)
scene_date = np.datetime_as_string(best_scene.time.values, unit='D')
print(f"-> Found scene from: {scene_date}")

# --- Step 2: Prepare the RGB data (same as before) ---
# The order B04, B03, B02 corresponds to Red, Green, Blue
rgb = best_scene[["B04", "B03", "B02"]].to_array(dim="band")

# --- Step 3: Scale the data for visualization (same as before) ---
rgb_scaled = rgb.clip(0, 3000) / 3000

Finding the clearest available scene...


/home/rsignell/miniforge3/envs/protocoast-notebook/lib/python3.13/site-packages/xarray/core/dataarray.py:6386: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmax(dim, axis, keep_attrs, skipna)


-> Found scene from: 2025-08-11


In [7]:
source_crs = f"{items[0].properties['proj:code']}"
source_crs

'EPSG:32619'

In [9]:
# --- Step 4: Plot with hvplot! ---
print("Generating interactive plot...")
# The .hvplot.rgb method is specifically designed for this task.
# The 'bands' argument tells it which dimension holds the R, G, B channels.
image_plot = rgb_scaled.hvplot.rgb(
    x='x',
    y='y',
    bands='band',
    rasterize=True,    # Essential for good performance with large images
    frame_width=600,
    crs=source_crs,
    tiles='OSM',
    geo=True,
    flip_yaxis=False,   # not needed as hvplot figures this out
    title=f"{scene_date}"
)

# In a Jupyter notebook, this will display the interactive plot automatically
image_plot

Generating interactive plot...


:DynamicMap   []
   :Overlay
      .WMTS.I :WMTS   [Longitude,Latitude]
      .RGB.I  :RGB   [x,y]   (R,G,B)